In [ ]:
import os
import boto3
import logging
import requests
import re
import datetime
import pandas as pd
import pytz
from boto3.s3.transfer import TransferConfig
from datetime import datetime, date
import base64
from datetime import datetime, timezone, timedelta
from io import StringIO

In [ ]:
current_datetime_gmt = datetime.now(timezone.utc)

print("GMT Current Time",current_datetime_gmt)

from_date = date.fromisoformat(current_datetime_gmt.date().isoformat()) - timedelta(days=1)
print("GMT Start Time" ,from_date)
to_date = date.fromisoformat(current_datetime_gmt.date().isoformat()) - timedelta(days=0)
print("GMT End Time",to_date)

In [ ]:
url = "https://data-eu.mixpanel.com/api/2.0/export/"
api_secret = "7cdc2d015ce13c26b1387a43f8200702"

event = None 
where = None  
print("From Date :",from_date)
print("To Date :",to_date)

end_time = datetime.utcnow()
start_time = end_time - timedelta(minutes=15)

params = {
    'from_date': start_time.strftime('%Y-%m-%d'),
    'to_date': end_time.strftime('%Y-%m-%d'),
    'from_time': int(start_time.timestamp()),
    'to_time': int(end_time.timestamp()),
    "event": event,
    "where": where
}
encoded_secret = base64.b64encode(api_secret.encode()).decode()
headers = {
    "Authorization": "Basic {}".format(encoded_secret),
    "accept": "text/plain"
}

response = requests.get(url, headers=headers, params=params)

In [ ]:
print(response.text[0:1000])

In [ ]:
response_df = pd.read_json(StringIO(response.text), lines=True)

In [ ]:
import logging
from datetime import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, StructType
from pyspark.sql.functions import col, current_timestamp, from_unixtime, to_date, lit, explode, to_json

In [ ]:
import os
os.environ["SPARK_HOME"] = "/home/hadoop/.local/lib/python3.9/site-packages/pyspark"  # Or wherever your Spark is
os.environ["PATH"] = os.environ["SPARK_HOME"] + "/bin:" + os.environ["PATH"]
S3_REGION = 'ap-south-1'

In [ ]:
spark = SparkSession.builder \
    .appName("mixpanel_batch_load") \
    .config("spark.jars", "/home/hadoop/agrim_cdp/common_jars/postgresql-42.2.24.jar") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
s3 = boto3.client('s3', region_name=S3_REGION)

In [ ]:
df = spark.read.option("header",True).csv("s3a://agrim-mixpanel-prod/Retailer_App_Production/Events_Data/Daily_Snapshots/2025-07-16/")

In [ ]:
df.printSchema()

In [31]:
# User inputs
start_date_str = "2025-01-01"  # Example
end_date_str = datetime.today().strftime("%Y-%m-%d")  # Today by default

# Convert to datetime objects
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

In [32]:
# Generate list of date strings in YYYY-MM-DD format
date_list = []
current_date = start_date

while current_date <= end_date:
    date_list.append(current_date.strftime("%Y-%m-%d"))
    current_date += timedelta(days=1)

In [33]:
base_path = "s3a://agrim-mixpanel-prod/Retailer_App_Production/Events_Data/Daily_Snapshots/"

# Example: s3://.../2025-07-16/Mix_Panel_Events.csv
paths = [f"{base_path}{date}/Mix_Panel_Events.csv" for date in date_list]

In [34]:
df = spark.read.option("header", True).csv(paths)

In [36]:
df.count()

56195167